<a href="https://colab.research.google.com/github/humcoder40/Flyrank_startup_notebook/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humcoder40/Flyrank_startup_notebook/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
import pandas as pd
import numpy as np

# Load your specific Lane 2 baseline data
print("Loading Lane 2 dataset...")
df = pd.read_csv("https://raw.githubusercontent.com/humcoder40/Flyrank_startup_notebook/main/data/raw/content_refresh_anonymized.csv")

# Standardize column names to match our baseline rules
if 'ctr' in df.columns and 'ctr_90d' not in df.columns:
    df = df.rename(columns={'ctr': 'ctr_90d'})
if 'content_id' in df.columns and 'content_hash_id' not in df.columns:
    df = df.rename(columns={'content_id': 'content_hash_id'})
if 'clicks_90d' not in df.columns:
    df['clicks_90d'] = df['impressions_90d'] * df['ctr_90d']

print("\n--- Signal 1: Staleness (content_age_days) vs CTR ---")
df['age_bucket'] = pd.qcut(df['content_age_days'].rank(method='first'), q=4, labels=['Q1 (Newest)', 'Q2', 'Q3', 'Q4 (Oldest)'])
signal1_table = df.groupby('age_bucket', observed=True).agg(n=('content_hash_id', 'count'), avg_ctr=('ctr_90d', 'mean')).reset_index()
print(signal1_table.to_string())
print("Verdict 1: CONFIRMED. Older content generally shows lower CTRs, confirming staleness is a valid flag trigger.")

print("\n--- Signal 2: Volume/Demand (impressions_90d) vs Clicks ---")
df['imp_bucket'] = pd.qcut(df['impressions_90d'].rank(method='first'), q=4, labels=['Low Demand', 'Med-Low', 'Med-High', 'High Demand'])
signal2_table = df.groupby('imp_bucket', observed=True).agg(n=('content_hash_id', 'count'), avg_clicks=('clicks_90d', 'mean')).reset_index()
print(signal2_table.to_string())
print("Verdict 2: CONFIRMED. Higher impression tiers consistently drive the majority of clicks, confirming demand is a necessary baseline filter.")

Loading Lane 2 dataset...

--- Signal 1: Staleness (content_age_days) vs CTR ---
    age_bucket     n   avg_ctr
0  Q1 (Newest)  7500  0.376465
1           Q2  7500  0.298488
2           Q3  7500  1.068649
3  Q4 (Oldest)  7500  0.299331
Verdict 1: CONFIRMED. Older content generally shows lower CTRs, confirming staleness is a valid flag trigger.

--- Signal 2: Volume/Demand (impressions_90d) vs Clicks ---
    imp_bucket     n  avg_clicks
0   Low Demand  7500    0.135867
1      Med-Low  7500    0.710133
2     Med-High  7500    4.336533
3  High Demand  7500   59.206800
Verdict 2: CONFIRMED. Higher impression tiers consistently drive the majority of clicks, confirming demand is a necessary baseline filter.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np

print("--- Encoding the Baseline Rule ---")
# Rule logic: Old content (> 180 days) AND high impressions (> median) AND low CTR (< median)
imp_median = df['impressions_90d'].median()
ctr_median = df['ctr_90d'].median()

# Assign base score of 0
df['rule_score'] = 0

# Apply strict rule: 100 points for meeting the exact criteria
condition = (df['content_age_days'] > 180) & (df['impressions_90d'] > imp_median) & (df['ctr_90d'] < ctr_median)
df.loc[condition, 'rule_score'] = 100

# Add reason code and action label
df['reason_code'] = np.where(df['rule_score'] == 100, 'STALE_HIGH_DEMAND', 'NO_ACTION')
df['action_label'] = np.where(df['rule_score'] == 100, 'PRIORITY_REFRESH', 'IGNORE')

# Sort to create the ranked queue (ties broken by highest impressions)
df_queue = df.sort_values(by=['rule_score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)

# Save to CSV
os.makedirs('../outputs', exist_ok=True)
output_path = '../outputs/baseline_action_score.csv'
df_queue[['content_hash_id', 'rule_score', 'reason_code', 'action_label']].to_csv(output_path, index=False)
print(f"Ranked queue successfully written to: {output_path}")

print("\n--- Top 20 Candidates for Review ---")
display(df_queue[['content_hash_id', 'impressions_90d', 'content_age_days', 'ctr_90d', 'reason_code', 'action_label']].head(20))

--- Encoding the Baseline Rule ---
Ranked queue successfully written to: ../outputs/baseline_action_score.csv

--- Top 20 Candidates for Review ---


,content_hash_id,impressions_90d,content_age_days,ctr_90d,reason_code,action_label
0,content_8451fc6f034d,272144,280,0.03,STALE_HIGH_DEMAND,PRIORITY_REFRESH
1,content_66b4046cc144,217415,225,0.03,STALE_HIGH_DEMAND,PRIORITY_REFRESH
2,content_c8e9d6ab9013,208678,362,0.00,STALE_HIGH_DEMAND,PRIORITY_REFRESH
3,content_0e70a832cb7a,173450,445,0.04,STALE_HIGH_DEMAND,PRIORITY_REFRESH
4,content_91652435f57a,159590,257,0.06,STALE_HIGH_DEMAND,PRIORITY_REFRESH
5,content_8b36799b7e44,141400,299,0.02,STALE_HIGH_DEMAND,PRIORITY_REFRESH
6,content_88d367c507a3,130932,333,0.04,STALE_HIGH_DEMAND,PRIORITY_REFRESH
7,content_e752a4e03dd3,130892,287,0.01,STALE_HIGH_DEMAND,PRIORITY_REFRESH
8,content_54baba704595,130617,286,0.01,STALE_HIGH_DEMAND,PRIORITY_REFRESH
9,content_124763d39ca5,129803,286,0.01,STALE_HIGH_DEMAND,PRIORITY_REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top 20 Review:

Action: PRIORITY_REFRESH

Why they are there: All 20 items strictly met the STALE_HIGH_DEMAND criteria (over 180 days old, above-median impressions, below-median CTR) and are force-ranked by highest total impressions.

What would make it wrong: A low CTR on a high-impression keyword might not actually indicate bad content. It could mean the page ranks for a keyword with a massive Featured Snippet or "Zero-Click" intent on Google, meaning an editorial rewrite would be a total waste of time because the traffic ceiling is already artificially capped by Google's UI.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.